In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import seaborn as sns

In [3]:
# Load the dataset
df = pd.read_csv('/Users/gaurav/workspace/datascience/boston_university/semester3/ms_datascience/data/homework_3.1.csv') 
df.head()

,time,value1,value2,value3
0,0,1.764052,1.883151,-0.369182
1,1,0.420157,-1.327759,-0.219379
2,2,1.018738,-1.230485,1.139660
3,3,2.300893,1.029397,0.715264
4,4,1.947558,-1.093123,0.720132


To test for discontinuities in time series data surrounding an event, you use an Interrupted Time Series Analysis (ITSA) framework using Ordinary Least Squares (OLS) linear regression.

![Interrupted Time Series Analysis](../../../images/ITSA.png)

In [4]:
# 2. Create the dummy variable for event at time = 50
df["event_50"] = (df["time"] >= 50).astype(int)

# 3. Add constant term for OLS
X = sm.add_constant(df[["time", "event_50"]])

results = []

# 4. Loop through each value column and fit OLS
for col in ["value1", "value2", "value3"]:
    model = sm.OLS(df[col], X).fit()

    beta_2 = model.params["event_50"]
    p_val = model.pvalues["event_50"]
    t_stat = model.tvalues["event_50"]

    results.append(
        {
            "Variable": col,
            "Jump_Magnitude_Beta2": beta_2,
            "t_statistic": t_stat,
            "p_value": p_val,
        }
    )

# 5. Convert to dataframe and sort by absolute t-statistic
res_df = pd.DataFrame(results)
res_df["abs_t_stat"] = res_df["t_statistic"].abs()
res_df = res_df.sort_values(by="abs_t_stat", ascending=False)

print(res_df.to_string(index=False))

top_var = res_df.iloc[0]["Variable"]
print(f"\nThe variable with the strongest discontinuity at time=50 is: {top_var}")

Variable  Jump_Magnitude_Beta2  t_statistic  p_value  abs_t_stat
  value3              1.767254     4.356868 0.000033    4.356868
  value1              0.850813     1.736745 0.085607    1.736745
  value2              0.682746     1.598864 0.113103    1.598864

The variable with the strongest discontinuity at time=50 is: value3


In [5]:
# 2. Create indicator and centered post-event time interaction term
df["event_50"] = (df["time"] >= 50).astype(int)
df["T_post"] = (df["time"] - 50) * df["event_50"]

# 3. Add intercept and feature matrix
X = sm.add_constant(df[["time", "event_50", "T_post"]])

results = []
    
# 4. Fit model for each value column
for col in ["value1", "value2", "value3"]:
    model = sm.OLS(df[col], X).fit()

    beta_3 = model.params["T_post"]
    p_val = model.pvalues["T_post"]
    t_stat = model.tvalues["T_post"]

    results.append(
        {
            "Variable": col,
            "Slope_Change_Beta3": beta_3,
            "t_statistic": t_stat,
            "p_value": p_val,
        }
    )

# 5. Convert to dataframe and sort by absolute t-statistic
res_df = pd.DataFrame(results)
res_df["abs_t_stat"] = res_df["t_statistic"].abs()
res_df = res_df.sort_values(by="abs_t_stat", ascending=False)
print(res_df.to_string(index=False))
top_var = res_df.iloc[0]["Variable"]
print(
    f"\nThe variable with the strongest derivative discontinuity at time=50 is: {top_var}"
)

Variable  Slope_Change_Beta3  t_statistic      p_value  abs_t_stat
  value1            0.105325     7.951259 3.602073e-12    7.951259
  value3            0.050695     3.857133 2.076610e-04    3.857133
  value2            0.036921     2.566732 1.181130e-02    2.566732

The variable with the strongest derivative discontinuity at time=50 is: value1


In [6]:
#To test whether the slope change is statistically significant when assuming zero level shift, 
# you can fit a model that includes only the time variable and the interaction term (T_post) without the event_50 dummy variable. 
# This will allow you to assess the significance of the slope change while controlling for the time trend.

# 2. Create the continuous piecewise term: max(0, time - 50)
df["T_post_continuous"] = np.maximum(0, df["time"] - 50)

# 3. Design matrix (Intercept + continuous time + slope change term)
X = sm.add_constant(df[["time", "T_post_continuous"]])

results = []

# 4. Run OLS for each value column
for col in ["value1", "value2", "value3"]:
    model = sm.OLS(df[col], X).fit()

    beta_3 = model.params["T_post_continuous"]
    t_stat = model.tvalues["T_post_continuous"]
    p_val = model.pvalues["T_post_continuous"]

    results.append(
        {
            "Variable": col,
            "Slope_Change_Beta3": beta_3,
            "t_statistic": t_stat,
            "p_value": p_val,
        }
    )

# 5. Rank variables by absolute t-statistic
res_df = pd.DataFrame(results)
res_df["abs_t_stat"] = res_df["t_statistic"].abs()
res_df = res_df.sort_values(by="abs_t_stat", ascending=False)

print(res_df.to_string(index=False))

strongest_var = res_df.iloc[0]["Variable"]
print(
    f"\nStrongest derivative discontinuity (without level jump): {strongest_var}"
)

Variable  Slope_Change_Beta3  t_statistic      p_value  abs_t_stat
  value1            0.104783     7.731065 9.959361e-12    7.731065
  value3            0.049620     3.418837 9.210393e-04    3.418837
  value2            0.036501     2.513997 1.358339e-02    2.513997

Strongest derivative discontinuity (without level jump): value1


In [7]:
'''
Step 3: Model Comparison (ANOVA / F-Test)
To verify whether forcing "no level jump" is a valid assumption compared to allowing both a jump and a slope change, 
run an ANOVA test comparing the restricted model (slope shift only) against the unrestricted model (level jump + slope shift):
'''
# Unrestricted model (includes level jump indicator D)
df["event_50"] = (df["time"] >= 50).astype(int)
X_unrestricted = sm.add_constant(df[["time", "event_50", "T_post_continuous"]])

model_restricted = sm.OLS(df["value1"], X).fit()
model_unrestricted = sm.OLS(df["value1"], X_unrestricted).fit()

# F-test for necessity of the level shift term
anova_results = sm.stats.anova_lm(
    model_restricted, model_unrestricted, typ=1
)
print(anova_results)
#If $p > 0.05$ on the F-test, omitting the level shift parameter is statistically justified.

   df_resid        ssr  df_diff   ss_diff         F   Pr(>F)
0      97.0  92.795521      0.0       NaN       NaN      NaN
1      96.0  87.696907      1.0  5.098614  5.581348  0.02017


To perform a Difference-in-Differences (DiD) analysis on homework_3.2.a.csv and homework_3.2.b.csv, here is the mathematical framework and Python/R code to run the regressions and answer Questions 3 through 5.

![Interrupted Time Series Analysis](../../../images/DiD.png)
Difference-in-Differences setup. Source: Medium

To determine which dataset has the largest treatment effect under the parallel trends assumption, calculate the **Difference-in-Differences estimator ($\hat{\beta}_3$)** for each dataset:

$$\hat{\beta}_3 = (\bar{Y}_{\text{Treat, After}} - \bar{Y}_{\text{Treat, Before}}) - (\bar{Y}_{\text{Control, After}} - \bar{Y}_{\text{Control, Before}})$$

The dataset with the **largest absolute value $\vert{}\hat{\beta}_3\vert{}$** (or the largest positive value if looking specifically for positive treatment magnitude) has the largest treatment effect.

---



### Step 1: Compute the Group Means for Each Dataset

For both `homework_3.2.a.csv` and `homework_3.2.b.csv`, calculate the 4 cell averages:

1. $\bar{Y}_{T, \text{after}}$: Mean outcome of Treatment group in "after" period.
2. $\bar{Y}_{T, \text{before}}$: Mean outcome of Treatment group in "before" period.
3. $\bar{Y}_{C, \text{after}}$: Mean outcome of Control group in "after" period.
4. $\bar{Y}_{C, \text{before}}$: Mean outcome of Control group in "before" period.

---

In [8]:
df_group1 = pd.read_csv('/Users/gaurav/workspace/datascience/boston_university/semester3/ms_datascience/data/homework_3.2.a.csv')
df_group1.head()

,group1,time1,outcome1
0,0,0,0.882026
1,0,1,1.600079
2,0,0,0.489369
3,0,1,2.520447
4,0,0,0.933779


In [9]:
df_group2 = pd.read_csv('/Users/gaurav/workspace/datascience/boston_university/semester3/ms_datascience/data/homework_3.2.b.csv')
df_group2.head()

,group2,time2,outcome2
0,0,0,0.667155
1,0,1,2.470969
2,0,0,-0.506778
3,0,1,1.525657
4,0,0,0.273664


In [10]:
def prepare_did_data(data, group_col, time_col, outcome_col):
    did_data = data.rename(
        columns={
            group_col: "treatment",
            time_col: "post",
            outcome_col: "value",
        }
    ).copy()
    did_data["treatment"] = did_data["treatment"].astype(int)
    did_data["post"] = did_data["post"].astype(int)
    return did_data


def run_did_analysis(data, name):
    model = smf.ols("value ~ treatment * post", data=data).fit()
    print(f"=== DiD Regression Results for {name} ===")
    print(model.summary())
    return model


df_group1_did = prepare_did_data(df_group1, "group1", "time1", "outcome1")
df_group2_did = prepare_did_data(df_group2, "group2", "time2", "outcome2")

model_a = run_did_analysis(df_group1_did, "Dataset A")
model_b = run_did_analysis(df_group2_did, "Dataset B")

did_a = model_a.params["treatment:post"]
did_b = model_b.params["treatment:post"]

print(f"Dataset A Treatment Effect (DiD): {did_a:.4f}")
print(f"Dataset B Treatment Effect (DiD): {did_b:.4f}")

if abs(did_a) > abs(did_b):
    print("-> homework_3.2.a.csv has the LARGER treatment effect.")
elif abs(did_b) > abs(did_a):
    print("-> homework_3.2.b.csv has the LARGER treatment effect.")
else:
    print("-> Both datasets have the exact same treatment effect size.")

=== DiD Regression Results for Dataset A ===
                            OLS Regression Results                            
Dep. Variable:                  value   R-squared:                       0.899
Model:                            OLS   Adj. R-squared:                  0.899
Method:                 Least Squares   F-statistic:                     2964.
Date:                Sat, 26 Sep 2026   Prob (F-statistic):               0.00
Time:                        13:31:59   Log-Likelihood:                -712.28
No. Observations:                1000   AIC:                             1433.
Df Residuals:                     996   BIC:                             1452.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------

### Key Decision Criteria

* **If you have the data or output numbers:** Compare $\hat{\beta}_{3, A}$ vs. $\hat{\beta}_{3, B}$.
* **If you are assessing via visual plots:** Look for the dataset where the **divergence between treatment and control trajectories post-treatment** is most pronounced relative to their pre-treatment baseline shift.

In the regression output, $\hat{\beta}_3$ is the coefficient on the `treatment:post` interaction term. The code also prints these values directly:

- Dataset A: $\hat{\beta}_{3,A} = 0.6858$
- Dataset B: $\hat{\beta}_{3,B} = 1.3499$

Because $|1.3499| > |0.6858|$, Dataset B (`homework_3.2.b.csv`) has the larger Difference-in-Differences treatment effect. The visual-plot criterion is an alternative way to compare the datasets; it is not a separate number produced by the regression output.

---

To determine which dataset has the most **statistically significant** treatment effect, you evaluate the **$t$-statistic** (or $p$-value) of the interaction coefficient $\hat{\beta}_3$ ($\text{treatment} \times \text{post}$) rather than just the magnitude of the coefficient itself.

The $t$-statistic measures how many standard errors the estimated effect is away from zero:

$$t = \frac{\hat{\beta}_3}{\text{SE}(\hat{\beta}_3)}$$

---



### Comparison Rule

1. **Highest Absolute $t$-statistic ($\vert{}t\vert{}$):** The dataset with the larger $\vert{}t_{\beta_3}\vert{}$ has the most statistically significant treatment effect (most evidence against the null hypothesis $H_0: \beta_3 = 0$).
2. **Smallest $p$-value:** Correspondingly, the dataset with the smallest $p$-value for $\hat{\beta}_3$ is the most statistically significant.

---

In [11]:
# 3. Extract key statistics for the interaction term (treatment:post)
t_a = model_a.tvalues["treatment:post"]
p_a = model_a.pvalues["treatment:post"]
se_a = model_a.bse["treatment:post"]
beta_a = model_a.params["treatment:post"]

t_b = model_b.tvalues["treatment:post"]
p_b = model_b.pvalues["treatment:post"]
se_b = model_b.bse["treatment:post"]
beta_b = model_b.params["treatment:post"]

print(
    f"Dataset A: Effect = {beta_a:.4f}, SE = {se_a:.4f}, t-stat = {t_a:.4f}, p-val = {p_a:.4e}"
)
print(
    f"Dataset B: Effect = {beta_b:.4f}, SE = {se_b:.4f}, t-stat = {t_b:.4f}, p-val = {p_b:.4e}"
)

# 4. Determine most statistically significant
if abs(t_a) > abs(t_b):
    print(
        "\n-> homework_3.2.a.csv has the MOST statistically significant treatment effect."
    )
elif abs(t_b) > abs(t_a):
    print(
        "\n-> homework_3.2.b.csv has the MOST statistically significant treatment effect."
    )
else:
    print("\n-> Both datasets have identical statistical significance.")


Dataset A: Effect = 0.6858, SE = 0.0625, t-stat = 10.9696, p-val = 1.6401e-26
Dataset B: Effect = 1.3499, SE = 0.1470, t-stat = 9.1798, p-val = 2.4324e-19

-> homework_3.2.a.csv has the MOST statistically significant treatment effect.


## Interpreting Treatment Effect Size Versus Statistical Significance

These two comparisons answer different questions:

1. **Treatment effect size:** Which dataset estimates the larger change caused by the treatment? Compare the DiD coefficients, $\hat{\beta}_3$. Dataset B has the larger estimated effect:

   $$\hat{\beta}_{3,A} = 0.6858, \qquad \hat{\beta}_{3,B} = 1.3499$$

   Therefore, `homework_3.2.b.csv` has the larger estimated treatment effect because $1.3499 > 0.6858$.

2. **Statistical significance:** Which dataset provides stronger evidence that its estimated effect is different from zero? Compare the absolute $t$-statistics or the $p$-values. Dataset A has the stronger statistical evidence:

   - Dataset A: Effect = `0.6858`, SE = `0.0625`, $t = 10.9696$, $p = 1.6401 \times 10^{-26}$
   - Dataset B: Effect = `1.3499`, SE = `0.1470`, $t = 9.1798$, $p = 2.4324 \times 10^{-19}$

   Dataset A has the larger absolute $t$-statistic (`10.9696 > 9.1798`) and the smaller $p$-value. Thus, `homework_3.2.a.csv` has the **most statistically significant** treatment effect, even though Dataset B has the **larger** treatment effect.

In short, Dataset B estimates a bigger effect, while Dataset A estimates its effect more precisely relative to its standard error. Both effects are statistically significant because both $p$-values are far below `0.05`.

## Simple Business Example

Imagine a company tests two online advertising campaigns using a Difference-in-Differences analysis. The company compares sales before and after each campaign and compares customers who saw the campaign with a control group that did not see it.

- **Campaign A** increases sales by `0.6858` units. Its standard error is small, so the estimate is very precise. This gives a high $t$-statistic and a very small $p$-value.
- **Campaign B** increases sales by `1.3499` units. This is a larger improvement, but the estimate is more variable, so its $t$-statistic is lower than Campaign A's.

The business conclusion depends on the question. If the company wants the **largest estimated sales improvement**, it should choose Campaign B. If it wants the result with the **strongest statistical evidence**, Campaign A is stronger. Statistical significance does not mean that Campaign A has the larger business impact; it means that its estimated impact is measured more precisely relative to its uncertainty.

To make this crystal clear, let's step away from unit sales and look at a business scenario where risk, sample size, and noise play a major role: **testing a new pricing strategy vs. launching a high-risk premium tier.**

---

### The Scenario: E-Commerce Subscription Business

An e-commerce company runs two simultaneous Difference-in-Differences (DiD) experiments to boost Monthly Recurring Revenue (MRR) per user.

* **Campaign A (Checkout Optimization):** A subtle UI tweak giving customers a 5% discount if they pre-pay annually.
* Tested on **100,000 users** (massive sample size, low noise).


* **Campaign B (VIP Premium Membership):** A bold new $50/month VIP tier offering free express shipping and exclusive perks.
* Tested on **2,000 users** (small sample size, high individual variability).



---

### Real-World Business Breakdown

#### 1. Treatment Effect Size ($\hat{\beta}_3$): *How big is the prize?*

* **Campaign A:** Increased average revenue per user by **$0.69**.
* **Campaign B:** Increased average revenue per user by **$1.35**.

> **Business Takeaway:** Campaign B is **twice as impactful in business scale**. If both campaigns cost the same to roll out globally, Campaign B generates vastly more revenue per customer.

---

#### 2. Statistical Significance ($t$-statistic / $p$-value): *How certain are we it wasn't a fluke?*

* **Campaign A ($t = 10.97$):** Tested on a huge sample with uniform user responses. The standard error is tiny ($\text{SE} = 0.06$). We are virtually $100\%$ certain that the $0.69 increase was caused by the feature, not random background noise.
* **Campaign B ($t = 9.18$):** Tested on a smaller sample where a few big spenders heavily swung the numbers. The standard error is higher ($\text{SE} = 0.15$). While the result is still overwhelmingly real ($p < 0.001$), there is more measurement noise surrounding the exact number.

---

### Summary Comparison Table

| Metric | Campaign A (UI Tweak) | Campaign B (VIP Tier) | Winner |
| --- | --- | --- | --- |
| **Estimated Lift ($\hat{\beta}_3$)** | +$0.69 / user | **+$1.35 / user** | **Campaign B** (Bigger Business Impact) |
| **Standard Error ($\text{SE}$)** | **0.06** (Very low noise) | 0.15 (Higher noise) | **Campaign A** |
| **$t$-Statistic** | **10.97** | 9.18 | **Campaign A** (Stronger Statistical Proof) |
| **Both $p$-values < 0.05?** | Yes | Yes | Both are real effects! |

---

### Executive Decision-Making: Which do you pick?

In practice, **you do NOT pick Campaign A just because its $t$-statistic is higher.**

Since *both* $p$-values are vastly smaller than 0.05, you have already cleared the hurdle proving neither result is random noise. Therefore, the executive decision comes down to **Economic Significance**:

1. **Choose Campaign B** if your goal is maximizing top-line revenue ($1.35 > $0.69).
2. **Recognize Campaign A's strength** as being extraordinarily predictable, but don't confuse "higher precision" with "higher revenue."

> **The Golden Rule:** Statistical significance tells you if an effect is **real**; effect size tells you if an effect **matters to the business**.

In [17]:
# Using formula syntax (automatically creates main effects and interaction)
df_group2_did = prepare_did_data(df_group2, "group2", "time2", "outcome2")
model = smf.ols("value ~ treatment + post + treatment:post", data=df_group2_did).fit()

# Extract the treatment effect (beta_3)
group2_treatment_effect = model.params["treatment:post"]
print(f"Treatment effect for Group 2: {group2_treatment_effect:.4f}")

Treatment effect for Group 2: 1.3499
